In [4]:
from config import Config
from DataManager import DataManager
from DataPreprocessor import Preprocessor
from FeatureSelection import FeatureSelector
from ModelManager import ModelManager
from ModelEvaluation import Evaluator

from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
cfg = Config()
data_manager = DataManager(cfg)
preprocessor = Preprocessor(cfg)

gt_df = data_manager.load_ground_truth()
wl = data_manager.load_wavelengths()

#image = data_manager.load_image(1200)
#processed = preprocessor.process_image(image)

X, y, valid_indices = preprocessor.process_dataset(data_manager, gt_df)
X.shape

(1732, 150)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=cfg.RANDOM_SEED)

In [ ]:
selector = FeatureSelector(cfg, n_features=25)
selector.fit(X_train, y_train)

X_train_rfe = selector.transform(X_train)
X_test_rfe = selector.transform(X_test)

In [21]:
models = ModelManager(cfg)

models.tune_fit(X_train_rfe, y_train, verbose=True)
y_pred = models.predict(X_test_rfe)
y_baseline = models.get_baseline_predictions(X_train=X_train_rfe, y_train=y_train, X_test=X_test_rfe)


Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params for [P] parameter: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 20}

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params for [K] parameter: {'n_estimators': 700, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 50}

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params for [Mg] parameter: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': None}

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params for [pH] parameter: {'n_estimators': 700, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 40}



In [22]:
evaluator = Evaluator(cfg)
results = evaluator.calculate_hyperview_score(y_test, y_pred, y_baseline)

print(results)
models.save_models(filename='best_rf_model.pkl')

P              0.971106
K              0.946097
Mg             0.905409
pH             0.844258
FINAL_SCORE    0.916717
dtype: float64
Successfully saved models to: ../Submissions\Models\best_rf_model.pkl!


In [16]:
models_new = ModelManager(cfg)
models_new.load_models()

y_pred_new = models_new.predict(X_test_rfe)
evaluator.calculate_hyperview_score(y_test, y_pred_new, y_baseline)

P              0.966009
K              0.945428
Mg             0.936602
pH             0.848595
FINAL_SCORE    0.924159
dtype: float64